In [17]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

print('ok')

ok


## SCRAPING

In [ ]:
#εδώ είναι το λινκ που περιέχει όλα τα events 
URL = "http://ufcstats.com/statistics/events/completed?page=all"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(URL, headers=headers)

In [11]:
response.status_code

200

In [12]:
soup = BeautifulSoup(response.text, "html.parser")

In [13]:
rows = soup.select("tr.b-statistics__table-row")

len(rows)

775

In [14]:
events = []

for row in rows:

    link = row.find("a")

    if link:

        event_name = link.text.strip()

        event_url = link["href"]

        cells = row.find_all("td")

        date = cells[0].text.strip()

        location = cells[1].text.strip()

        events.append({
            "event_name": event_name,
            "event_url": event_url,
            "date": date,
            "location": location
        })

In [15]:
events

[{'event_name': 'UFC 328: Chimaev vs. Strickland',
  'event_url': 'http://ufcstats.com/event-details/9eedac48b497de5a',
  'date': 'UFC 328: Chimaev vs. Strickland\n                        \n\n                          May 09, 2026',
  'location': 'Newark, New Jersey, USA'},
 {'event_name': 'UFC Fight Night: Della Maddalena vs. Prates',
  'event_url': 'http://ufcstats.com/event-details/872b018076f831b0',
  'date': 'UFC Fight Night: Della Maddalena vs. Prates\n                        \n\n                          May 02, 2026',
  'location': 'Perth, Western Australia, Australia'},
 {'event_name': 'UFC Fight Night: Sterling vs. Zalal',
  'event_url': 'http://ufcstats.com/event-details/e60d773a0a42048a',
  'date': 'UFC Fight Night: Sterling vs. Zalal\n                        \n\n                          April 25, 2026',
  'location': 'Las Vegas, Nevada, USA'},
 {'event_name': 'UFC Fight Night: Burns vs. Malott',
  'event_url': 'http://ufcstats.com/event-details/c3ac8d0da7b05772',
  'date'

In [16]:
len(events)

773

In [19]:
df_events = pd.DataFrame(events)

## CLEANING


In [24]:
df_events.head()

,event_name,event_url,date,location
0,UFC 328: Chimaev vs. Strickland,http://ufcstats.com/event-details/9eedac48b497...,UFC 328: Chimaev vs. Strickland\n ...,"Newark, New Jersey, USA"
1,UFC Fight Night: Della Maddalena vs. Prates,http://ufcstats.com/event-details/872b018076f8...,UFC Fight Night: Della Maddalena vs. Prates\n ...,"Perth, Western Australia, Australia"
2,UFC Fight Night: Sterling vs. Zalal,http://ufcstats.com/event-details/e60d773a0a42...,UFC Fight Night: Sterling vs. Zalal\n ...,"Las Vegas, Nevada, USA"
3,UFC Fight Night: Burns vs. Malott,http://ufcstats.com/event-details/c3ac8d0da7b0...,UFC Fight Night: Burns vs. Malott\n ...,"Winnipeg, Manitoba, Canada"
4,UFC 327: Prochazka vs. Ulberg,http://ufcstats.com/event-details/f3eb664db7fb...,UFC 327: Prochazka vs. Ulberg\n ...,"Miami, Florida, USA"


In [ ]:
df_events['date'][100] ## βλέπω ότι δεν τα πέρνει σωστά επομένως πρέπει να τα περιορίσω 

'UFC 296: Edwards vs. Covington\n                        \n\n                          December 16, 2023'

In [ ]:
## τεστάρω να δω τι θα κόψω
## θα πάρω τα τελευταία 18 στοιχία καθώς ο μεγαλύτερος μήνας έχει 9 στοιχία + 9 πάμε μαξ 18 

for i in range(df_events.shape[0]):
    print(str(df_events['date'][i])[-18:].strip())
    if i > 10 :
        break

May 09, 2026
May 02, 2026
April 25, 2026
April 18, 2026
April 11, 2026
April 04, 2026
March 28, 2026
March 21, 2026
March 14, 2026
March 07, 2026
February 28, 2026
February 21, 2026


In [46]:
## αλλάζω το date

df_events['date'] = df_events['date'].astype(str).str[-18:].str.strip()

In [49]:
df_events.head(5)

,event_name,event_url,date,location
0,UFC 328: Chimaev vs. Strickland,http://ufcstats.com/event-details/9eedac48b497...,"May 09, 2026","Newark, New Jersey, USA"
1,UFC Fight Night: Della Maddalena vs. Prates,http://ufcstats.com/event-details/872b018076f8...,"May 02, 2026","Perth, Western Australia, Australia"
2,UFC Fight Night: Sterling vs. Zalal,http://ufcstats.com/event-details/e60d773a0a42...,"April 25, 2026","Las Vegas, Nevada, USA"
3,UFC Fight Night: Burns vs. Malott,http://ufcstats.com/event-details/c3ac8d0da7b0...,"April 18, 2026","Winnipeg, Manitoba, Canada"
4,UFC 327: Prochazka vs. Ulberg,http://ufcstats.com/event-details/f3eb664db7fb...,"April 11, 2026","Miami, Florida, USA"


In [50]:
## θα πάω τωρα να σπάσω το date σε μήνα - μέρα - έτος  το location σε πόλη και χώρα και το event_name σε fight_night ή κανονικό ή άλλο

In [56]:
## πρώτο !

month = []
day = []
year = []
for i in range(df_events.shape[0]):
    broken = df_events['date'][i].split(' ')
    month.append(broken[0])
    day.append(str(broken[1]).replace(',',''))
    year.append(broken[2])


In [57]:
df_events['day'] = day
df_events['month'] = month
df_events['year'] = year

In [59]:
df_events.drop(columns=['date'],inplace=True)

In [71]:
##2o !!
## κάποιες τοποθεσίες δεν έχουν  ρεγιον 
city=[]
region = []
country = []

for i in range(df_events.shape[0]):
    broken = df_events['location'][i].split(',')
    if len(broken) == 3:
        city.append(broken[0])
        region.append(broken[1])
        country.append(broken[2])


    if len(broken) <3 :
        city.append(broken[0])
        country.append(broken[1])
        region.append('unknown')
    


In [73]:
df_events['city'] = city
df_events['region'] = region
df_events['country'] = country



In [75]:
df_events.drop(columns=['location'],inplace=True)

In [76]:
df_events.head(2)

,event_name,event_url,day,month,year,city,region,country
0,UFC 328: Chimaev vs. Strickland,http://ufcstats.com/event-details/9eedac48b497...,09,May,2026,Newark,New Jersey,USA
1,UFC Fight Night: Della Maddalena vs. Prates,http://ufcstats.com/event-details/872b018076f8...,02,May,2026,Perth,Western Australia,Australia


In [81]:
card = []
for i in range(df_events.shape[0]):
    if 'UFC Fight Night' in df_events['event_name'][i]:
        card.append(0)
    else:
        card.append(1)
card

df_events['main_card'] = card


In [83]:
df_events.head(10)

,event_name,event_url,day,month,year,city,region,country,main_card
0,UFC 328: Chimaev vs. Strickland,http://ufcstats.com/event-details/9eedac48b497...,09,May,2026,Newark,New Jersey,USA,1
1,UFC Fight Night: Della Maddalena vs. Prates,http://ufcstats.com/event-details/872b018076f8...,02,May,2026,Perth,Western Australia,Australia,0
2,UFC Fight Night: Sterling vs. Zalal,http://ufcstats.com/event-details/e60d773a0a42...,25,April,2026,Las Vegas,Nevada,USA,0
3,UFC Fight Night: Burns vs. Malott,http://ufcstats.com/event-details/c3ac8d0da7b0...,18,April,2026,Winnipeg,Manitoba,Canada,0
4,UFC 327: Prochazka vs. Ulberg,http://ufcstats.com/event-details/f3eb664db7fb...,11,April,2026,Miami,Florida,USA,1
5,UFC Fight Night: Moicano vs. Duncan,http://ufcstats.com/event-details/9a70f67ad218...,04,April,2026,Las Vegas,Nevada,USA,0
6,UFC Fight Night: Adesanya vs. Pyfer,http://ufcstats.com/event-details/5c38639f860a...,28,March,2026,Seattle,Washington,USA,0
7,UFC Fight Night: Evloev vs. Murphy,http://ufcstats.com/event-details/69108cb8b32e...,21,March,2026,London,England,United Kingdom,0
8,UFC Fight Night: Emmett vs. Vallejos,http://ufcstats.com/event-details/babc6b574533...,14,March,2026,Las Vegas,Nevada,USA,0
9,UFC 326: Holloway vs. Oliveira 2,http://ufcstats.com/event-details/15ec018d1447...,07,March,2026,Las Vegas,Nevada,USA,1


In [91]:
## θα ορίσω ένα μοναδικό κωδικό σε κάθε ένα event ώστε αργότερα να κάνω join  με τους αγώνες που έγιναν στο καθένα,στην προκειμένη τον αριθμό Α/Α

df_events['ID'] = [i for i in range(df_events.shape[0],0,-1)]

In [92]:
df_events = df_events[['ID','event_name','main_card','year','country','month','day','city','region','event_url']]

In [94]:
df_events.tail(10)

,ID,event_name,main_card,year,country,month,day,city,region,event_url
763,10,UFC 10: The Tournament,1,1996,USA,July,12,Birmingham,Alabama,http://ufcstats.com/event-details/aee8eecfc4bf...
764,9,UFC 9: Motor City Madness,1,1996,USA,May,17,Detroit,Michigan,http://ufcstats.com/event-details/a390eb8a9b2d...
765,8,UFC 8: David vs Goliath,1,1996,Puerto Rico,February,16,San Juan,unknown,http://ufcstats.com/event-details/b63e800c18e0...
766,7,UFC - Ultimate Ultimate '95,1,1995,USA,December,16,Denver,Colorado,http://ufcstats.com/event-details/31bbd46d57df...
767,6,UFC 7: The Brawl in Buffalo,1,1995,USA,September,08,Buffalo,New York,http://ufcstats.com/event-details/5af480a3b2e1...
768,5,UFC 6: Clash of the Titans,1,1995,USA,July,14,Casper,Wyoming,http://ufcstats.com/event-details/1c3f5e85b59e...
769,4,UFC 5: The Return of the Beast,1,1995,USA,April,07,Charlotte,North Carolina,http://ufcstats.com/event-details/dedc3bb440d0...
770,3,UFC 4: Revenge of the Warriors,1,1994,USA,December,16,Tulsa,Oklahoma,http://ufcstats.com/event-details/b60391da771d...
771,2,UFC 3: The American Dream,1,1994,USA,September,09,Charlotte,North Carolina,http://ufcstats.com/event-details/1a49e0670dfa...
772,1,UFC 2: No Way Out,1,1994,USA,March,11,Denver,Colorado,http://ufcstats.com/event-details/a6a9ab5a824e...


In [95]:
df_events.to_csv('events.csv',index=False)